# 53. 直方图（histplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 10 / 20 步：读懂连续变量的整体分布**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 蜂群图（swarmplot）  →  **本章任务：** 直方图（histplot）  →  **下一步：** 核密度图（kdeplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

拿到销售额、身高这类连续数值，只看几个统计量很难看清“大部分数据落在哪”。



## 本章目标

学完本章，你将能够：

- **理解**：理解「直方图（histplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「直方图（histplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「直方图（histplot）」并读出其中的结论。


## 53.1 适用场景

**背景引入**：拿到销售额、身高这类连续数值，只看几个统计量很难看清“大部分数据落在哪”。直方图把数值按区间切成一根根柱子，一眼就能看出分布是集中、偏斜还是多峰，是理解数据分布最直观的起点。当还想比较不同分组（比如不同品类的价格）时，histplot 的 hue 参数能叠着看，省去反复切子集对比的麻烦。

探索连续变量分布，并需要通过hue比较类别。


## 53.2 数据结构

连续数值列，可配合一列分类变量。


## 53.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 bins 参数从 18 改为 12 或 25，观察分箱数量对分布细节的影响
2. 修改 stat="density" 为 stat="probability"，对比密度与概率的纵轴含义
3. 调整 multiple="fill" 为 multiple="stack"，说明填充与堆积对组间比较的差异


## 53.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`sns.histplot()`、`ax.set()`、`fig.tight_layout()` | 探索连续变量分布，并需要通过hue比较类别。 | 不同组样本量不等却比较频数 |
| 进阶变体 | `plt.subplots()`、`sns.histplot()`、`ax.set()`、`fig.tight_layout()` | 在基础图表上增加分组、注释、布局或交互 | 堆积后难以看清小组形状 |
| 关键参数 | `bins/binwidth` | 分箱 | 不同组样本量不等却比较频数 |
| 关键参数 | `stat` | 频数或密度 | 堆积后难以看清小组形状 |
| 关键参数 | `multiple` | layer/stack/fill | 同时打开过多视觉选项 |
| 关键参数 | `element` | bars/step/poly | 不同组样本量不等却比较频数 |


## 53.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-53 -->
### 数学推导｜直方图的频数与密度

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜用箱边界划分数轴。** 第 $j$ 个箱为 $[b_j,b_{j+1})$，箱宽 $h_j=b_{j+1}-b_j$。

**第 2 步｜数落入箱中的样本。** $n_j=\sum_i\mathbf{1}(b_j\le x_i<b_{j+1})$，相对频率为 $n_j/n$。

**第 3 步｜让柱形面积代表概率。** 柱高应满足“高 × 宽 = 相对频率”，所以

$$
\hat f_jh_j=\frac{n_j}{n}
\quad\Longrightarrow\quad
\hat f_j=\frac{n_j}{nh_j}
$$

把所有柱面积相加就得到 1。

**把上面的关系收束为本章计算式：**

$$
\hat{f}_j=\frac{n_j}{n\,h_j}
$$

**符号解释：** $n_j$ 是第 $j$ 个箱中的样本数，$h_j$ 是箱宽；密度直方图总面积为 1。

**代码对应：** 固定 `bins` 或箱边界比较不同组；密度口径使用 `stat='density'` 或对应参数。

**使用边界：** 箱宽改变会显著改变形状；不同样本量的组不宜直接比较原始频数。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(f"Diamonds {len(diamonds):,} | Taxis {len(taxis):,} | Flights {len(flights):,} 行")


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(f"样本：orders {len(orders):,} | marketing {len(marketing):,} | daily {len(daily):,} 行")


## 53.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.3))
sns.histplot(data=orders, x="order_value", bins=18, color="#1a73e8", ax=ax)
ax.set(title="订单金额直方图", xlabel="客单价（元）", ylabel="订单数")
fig.tight_layout()
plt.show()


**练一练**：光看不练记不牢。这一节让你亲手改一次直方图的分箱参数，观察分布细节随分箱数量的变化。任务是把 `bins=18` 改成 `bins=12`，重新绘制订单金额直方图。分箱变少后条柱更宽、更少，右侧长尾的整体形态仍然保留，但局部抖动会被压缩；反过来，`bins` 太大又会把样本的偶然起伏放大。请先运行一次记住 `bins=18` 的样子，再改成 `bins=12` 运行，观察条柱数量的变化。完成后再运行自检确认。


In [ ]:
# 请在下方填写代码
# 练一练：把 bins 从 18 改为 12，重新绘制订单金额直方图并观察变化。
# 提示：复用上方基础图表，只修改 bins 参数，其余保持不变。
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# 完整参考答案
import matplotlib.pyplot as plt
import seaborn as sns

# 参考实现：只改变 bins，其余与基础图表保持一致
fig, ax = plt.subplots(figsize=(8, 4.3))
sns.histplot(data=orders, x="order_value", bins=12, color="#1a73e8", ax=ax)
ax.set(title="订单金额直方图（bins=12）", xlabel="客单价（元）", ylabel="订单数")
fig.tight_layout()
plt.show()


## 53.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.histplot(
    data=orders,
    x="order_value",
    hue="category",
    bins=18,
    stat="density",
    common_norm=False,
    element="step",
    fill=False,
    palette="colorblind",
    ax=ax,
)
ax.set(title="品类客单价密度对比", xlabel="客单价（元）", ylabel="密度")
fig.tight_layout()
plt.show()


## 53.8 参数说明

- bins/binwidth：分箱
- stat：频数或密度
- multiple：layer/stack/fill
- element：bars/step/poly


## 53.9 结果解读

比较中心、偏态和尾部；相同分箱边界是组间比较前提。


## 53.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 53.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 53.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 53.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 53.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 53.12 易错点提醒

- 不同组样本量不等却比较频数
- 堆积后难以看清小组形状
- 同时打开过多视觉选项


## 53.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 53.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：按渠道分色直方图，观察重叠分布的构成
# 【目标】用颜色分层同一变量的分布，看各渠道的重叠程度。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：加 hue="channel"，用半透明色叠加各渠道的分布。
fig, ax = plt.subplots(figsize=(8, 4.3))
sns.histplot(
    data=orders, x="order_value", hue="channel", bins=18, alpha=0.6, ax=ax
)
ax.set(title="分渠道订单金额分布", xlabel="客单价（元）", ylabel="订单数")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()

# ---- 反思记录：分层直方图下，各渠道分布重叠了吗 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.histplot(
    data=marketing,
    x="conversion",
    hue="channel",
    bins=16,
    multiple="fill",
    palette="colorblind",
    ax=ax,
)
ax.set(title="不同转化率区间的渠道构成", xlabel="转化率", ylabel="渠道构成比例")
fig.tight_layout()
plt.show()


## 53.15 小结

用histplot在Seaborn中完成分组、堆积和密度直方图。


### 53.15.1 你已经掌握

- 判断直方图（histplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 53.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `bins/binwidth` | 分箱 |
| `stat` | 频数或密度 |
| `multiple` | layer/stack/fill |
| `element` | bars/step/poly |


### 53.15.3 需要注意

- 不同组样本量不等却比较频数
- 堆积后难以看清小组形状
- 同时打开过多视觉选项


### 53.15.4 完成检查

- [ ] 能判断什么问题适合使用直方图（histplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 53.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
